# Sentiment Analysis Pipeline for Unstructured Textual Data
# Using Large Language Models

**Author:** Wenlan (Tony) Xie  
**Affiliation:** The University of Sydney  
**Contact:** Wenlan Tony Xie ([wxie3035@uni.sydney.edu.au](mailto:wxie3035@uni.sydney.edu.au)) 
**Last Updated:** January 2026

---

**Last Updated:** January 2026 | **Python Version:** 3.10+ | **License:** MIT

---

### Associated Publication

> Tian, H., Xie, W. T., & Zhang, Y. (2026). Reading Between the Reels: An AI-Driven Approach to
> Analysing Movie Review Sentiment and Market Returns.
> *International Journal of Finance & Economics*.
> DOI: [10.1002/ijfe.70129](https://doi.org/10.1002/ijfe.70129)

### Data & Code Availability

The complete dataset and scraping scripts are publicly available at:
[https://github.com/WLXie-Tony/Movie_Review_Analysis](https://github.com/WLXie-Tony/Movie_Review_Analysis)

---

## Table of Contents

1. [Overview & Motivation](#1-overview--motivation)
2. [Environment Setup & Dependencies](#2-environment-setup--dependencies)
3. [Data Schema Definition](#3-data-schema-definition)
4. [Formal Problem Specification](#4-formal-problem-specification)
5. [Pipeline Implementation](#5-pipeline-implementation)
    - 5.1 [Configuration](#51-configuration)
    - 5.2 [Core Pipeline Class](#52-core-pipeline-class)
6. [Execution & Demonstration](#6-execution--demonstration)
7. [Notes on Reproducibility](#7-notes-on-reproducibility)

---

## 1. Overview & Motivation

This notebook implements a production-grade **ETL (Extract, Transform, Load) pipeline** that converts approximately 247,850 unstructured movie reviews into structured sentiment measures suitable for econometric analysis. These measures serve as the primary independent variables in the associated asset pricing study.

**Design objectives.** The pipeline is engineered around four principles:

| Principle | Implementation |
|:---|:---|
| **Scalability** | Asynchronous concurrency via `asyncio` semaphores, reducing wall-clock time by ~95% relative to synchronous execution |
| **Reproducibility** | Pinned model version (`gpt-4o-2024-05-13`), low temperature (0.2), deterministic schema enforcement |
| **Data integrity** | Strict `Pydantic` validation on all LLM outputs; type-checked fields with bounded ranges |
| **Fault tolerance** | Truncated exponential backoff with jitter (via `tenacity`); idempotent checkpointing for resumable execution |

---

## 2. Environment Setup & Dependencies

The pipeline requires the following packages. All dependencies are available via `pip`:

```
pip install openai pandas tiktoken pydantic tenacity tqdm python-dotenv openpyxl
```

**API key configuration.** Store your OpenAI API key in a `.env` file at the project root:
```
OPENAI_API_KEY=sk-...
```

In [ ]:
# =============================================================================
# Section 2: Environment Setup & Dependencies
# =============================================================================

# Standard library
import os
import asyncio
import json
import logging
import warnings
from datetime import datetime
from typing import Dict, List, Optional

# Third-party: data manipulation
import pandas as pd
import tiktoken

# Third-party: API client
from dotenv import load_dotenv
from openai import AsyncOpenAI

# Third-party: resilience & validation
from pydantic import BaseModel, Field
from tenacity import (
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

# Third-party: progress monitoring
from tqdm.asyncio import tqdm

# ---------------------------------------------------------------------------
# Runtime configuration
# ---------------------------------------------------------------------------
warnings.filterwarnings("ignore")
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("SentimentPipeline")

logger.info("Environment configured. All dependencies loaded successfully.")

---

## 3. Data Schema Definition

To ensure that stochastic LLM outputs conform to a fixed structure required by downstream regression models (OLS, Fama-French, etc.), we enforce a strict `Pydantic` schema on every API response. Any response that violates the schema triggers a `ValidationError` and is automatically retried.

**Schema fields:**

| Field | Type | Constraints | Description |
|:------|:-----|:------------|:------------|
| `sentiment_score` | `int` | $\in [1, 10]$ | Overall sentiment polarity |
| `emotion_keywords` | `List[str]` | length $\in [1, 5]$ | Salient emotional terms |
| `primary_emotion` | `str` | -- | Dominant emotion category |
| `review_focus` | `str` | -- | Thematic focus (e.g., plot, acting) |
| `bias_analysis` | `str` | -- | Assessment of reviewer bias |
| `summary` | `str` | $\leq 50$ words | Concise review summary |

In [ ]:
# =============================================================================
# Section 3: Data Schema Definition (Pydantic v2)
# =============================================================================


class ReviewAnalysis(BaseModel):
    """Strict output schema for LLM-based sentiment extraction.

    Enforces type constraints and value bounds to guarantee data integrity
    for downstream econometric modeling (see Tian, Xie & Zhang, 2026, Sec. 3.2).
    """

    sentiment_score: int = Field(
        ...,
        ge=1,
        le=10,
        description="Integer sentiment score (1 = extremely negative, 10 = extremely positive)",
    )
    emotion_keywords: List[str] = Field(
        ...,
        min_length=1,
        max_length=5,
        description="1-5 keywords capturing the emotional tone of the review",
    )
    primary_emotion: str = Field(
        ...,
        description="Dominant emotion expressed (e.g., admiration, disappointment)",
    )
    review_focus: str = Field(
        ...,
        description="Primary thematic focus (e.g., plot, acting, cinematography)",
    )
    bias_analysis: str = Field(
        ...,
        description="Assessment of potential reviewer biases or subjective factors",
    )
    summary: str = Field(
        ...,
        description="Concise summary of the review (50 words or fewer)",
    )


logger.info(f"Schema defined: {len(ReviewAnalysis.model_fields)} validated fields.")

---

## 4. Formal Problem Specification

We formalize the LLM-based extraction process as a probabilistic mapping from unstructured text to a structured feature space, subject to cost and reliability constraints.

### 4.1 Sentiment Extraction as Conditional Generation

Let $\mathcal{D} = \{(T_i, \mathbf{X}_i)\}_{i=1}^{N}$ denote the corpus of $N$ movie reviews, where $T_i$ is the raw text and $\mathbf{X}_i \in \mathbb{R}^d$ is the associated metadata vector (budget, box office revenue, director, etc.). The LLM defines a parameterized conditional distribution $P_\theta(\cdot)$. For each review $i$, the structured output $\mathcal{S}_i$ is drawn as:

$$
\mathcal{S}_i \sim P_\theta\!\left(\mathcal{S} \;\middle|\; T_i \oplus \mathbf{X}_i,\; \mathcal{P};\; \tau\right)
$$

where $\mathcal{P}$ denotes the system prompt encoding domain constraints, $\tau = 0.2$ is the temperature hyperparameter set to minimize output entropy $H(P_\theta)$, and $\oplus$ denotes string concatenation.

### 4.2 Cost Model

Given the scale of the pipeline ($N \approx 2.5 \times 10^5$), cost management is essential. Let $\mathcal{T}(\cdot)$ denote the `cl100k_base` tokenizer. The total API cost is:

$$
C_{\text{total}} = \sum_{i=1}^{N} \left[ \frac{|\mathcal{T}(T_i \oplus \mathbf{X}_i \oplus \mathcal{P})|}{1000} \cdot \lambda_{\text{in}} \;+\; \frac{|\mathcal{T}(\mathcal{S}_i)|}{1000} \cdot \lambda_{\text{out}} \right]
$$

where $\lambda_{\text{in}}$ and $\lambda_{\text{out}}$ are the per-1K-token prices for input and output contexts, respectively.

### 4.3 Retry Strategy: Truncated Exponential Backoff with Jitter

To handle transient API failures (HTTP 429, 500, 503), the wait time before the $k$-th retry is:

$$
W_k = \min\!\left(W_{\text{cap}},\; W_{\text{base}} \cdot 2^k\right) + \epsilon, \qquad \epsilon \sim \mathrm{Uniform}(0, 1)
$$

where $W_{\text{base}} = 2\text{s}$, $W_{\text{cap}} = 60\text{s}$, and the jitter term $\epsilon$ decorrelates concurrent retries to prevent thundering-herd effects.

---

## 5. Pipeline Implementation

### 5.1 Configuration

All tunable hyperparameters are centralized in a single configuration class to facilitate experiment tracking and ensure reproducibility across runs.

In [ ]:
# =============================================================================
# Section 5: Pipeline Implementation
# =============================================================================


# ---------------------------------------------------------------------------
# 5.1 Configuration
# ---------------------------------------------------------------------------
class PipelineConfig:
    """Centralized configuration for the sentiment extraction pipeline.

    Notes
    -----
    - MODEL_NAME is pinned to a specific snapshot to ensure reproducibility.
    - Pricing reflects OpenAI rates as of January 2026; update if rates change.
    """

    # Model
    MODEL_NAME: str = "gpt-4o-2024-05-13"
    TEMPERATURE: float = 0.2

    # Concurrency & resilience
    MAX_CONCURRENCY: int = 20
    MAX_RETRIES: int = 5
    BATCH_SIZE: int = 50

    # Cost (USD per 1K tokens)
    COST_INPUT_PER_1K: float = 0.0050
    COST_OUTPUT_PER_1K: float = 0.0150


# ---------------------------------------------------------------------------
# 5.2 Core Pipeline Class
# ---------------------------------------------------------------------------
class MovieReviewResearcher:
    """Asynchronous ETL pipeline for LLM-based sentiment extraction.

    This class orchestrates the full lifecycle of the extraction process:
    data ingestion, concurrent API calls with rate limiting, schema validation,
    cost tracking, and idempotent checkpointing.

    Parameters
    ----------
    input_file : str
        Path to the input dataset (.xlsx or .csv).
    output_dir : str
        Directory for storing output CSV files.

    Attributes
    ----------
    total_cost : float
        Cumulative API cost in USD, updated atomically via an async lock.
    """

    # System prompt (constant across all requests)
    SYSTEM_PROMPT: str = (
        "You are a professional film critic and sentiment analysis expert. "
        "Analyze the following movie review and provide a detailed sentiment "
        "analysis. Return ONLY a JSON object conforming to the specified schema."
    )

    def __init__(self, input_file: str, output_dir: str) -> None:
        self.input_file = input_file
        self.output_dir = output_dir
        self.total_cost: float = 0.0

        # Concurrency primitives
        self._cost_lock = asyncio.Lock()
        self._semaphore = asyncio.Semaphore(PipelineConfig.MAX_CONCURRENCY)

        # Tokenizer for cost estimation
        try:
            self._tokenizer = tiktoken.encoding_for_model("gpt-4o")
        except KeyError:
            self._tokenizer = tiktoken.get_encoding("cl100k_base")

        # Async OpenAI client
        self._client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

        # Ensure output directory exists
        os.makedirs(self.output_dir, exist_ok=True)

    # -- Private helpers ----------------------------------------------------

    @staticmethod
    def _estimate_cost(prompt_tokens: int, completion_tokens: int) -> float:
        """Compute the USD cost of a single API request."""
        return (
            (prompt_tokens / 1_000) * PipelineConfig.COST_INPUT_PER_1K
            + (completion_tokens / 1_000) * PipelineConfig.COST_OUTPUT_PER_1K
        )

    def _build_user_prompt(self, row: pd.Series) -> str:
        """Construct the user-facing prompt from a data record.

        Separates prompt construction from API logic to facilitate unit testing.
        """
        return (
            f"Movie Information:\n"
            f"  Title: {row.get('Title', 'N/A')}\n"
            f"  Director: {row.get('Director', 'N/A')}\n"
            f"  Writers: {row.get('Writers', 'N/A')}\n"
            f"  Release Year: {row.get('Release_Year', 'N/A')}\n"
            f"  Budget: ${row.get('Budget', 0):,.0f}\n"
            f"  Opening Weekend (US/Canada): ${row.get('Opening_Weekend', 0):,.0f}\n"
            f"  Worldwide Gross: ${row.get('Gross_Worldwide', 0):,.0f}\n"
            f"  IMDb Rating: {row.get('IMDb_Rating', 'N/A')}/10\n"
            f"  Language: {row.get('Language', 'N/A')}\n"
            f"  Country: {row.get('Country', 'N/A')}\n"
            f"  Production Companies: {row.get('Production_Companies', 'N/A')}\n\n"
            f"Review Text:\n"
            f"\'\'\'{row.get('Comments', '')}\'\'\'"
        )

    # -- Core extraction (single record) ------------------------------------

    @retry(
        wait=wait_exponential(multiplier=1, min=2, max=60),
        stop=stop_after_attempt(PipelineConfig.MAX_RETRIES),
        retry=retry_if_exception_type(Exception),
        reraise=True,
    )
    async def _analyze_single_row(
        self, idx: int, row: pd.Series
    ) -> Optional[Dict]:
        """Extract structured sentiment from a single review.

        This method is the atomic unit of the pipeline. It is wrapped with
        a retry decorator implementing truncated exponential backoff and
        executed within a semaphore context to enforce concurrency limits.

        Parameters
        ----------
        idx : int
            Row index in the source DataFrame (used for idempotency).
        row : pd.Series
            A single record containing review text and metadata.

        Returns
        -------
        dict or None
            Enriched record with extracted sentiment fields and cost metadata,
            or None if all retry attempts are exhausted.
        """
        async with self._semaphore:
            try:
                # API call with enforced JSON output mode
                response = await self._client.chat.completions.create(
                    model=PipelineConfig.MODEL_NAME,
                    messages=[
                        {"role": "system", "content": self.SYSTEM_PROMPT},
                        {"role": "user", "content": self._build_user_prompt(row)},
                    ],
                    response_format={"type": "json_object"},
                    temperature=PipelineConfig.TEMPERATURE,
                )

                # Schema validation (raises ValidationError on mismatch)
                raw_json = response.choices[0].message.content
                parsed = ReviewAnalysis.model_validate_json(raw_json)

                # Thread-safe cost accumulation
                usage = response.usage
                cost = self._estimate_cost(usage.prompt_tokens, usage.completion_tokens)
                async with self._cost_lock:
                    self.total_cost += cost

                return {
                    "original_index": idx,
                    **row.to_dict(),
                    **parsed.model_dump(),
                    "request_cost_usd": round(cost, 6),
                    "prompt_tokens": usage.prompt_tokens,
                    "completion_tokens": usage.completion_tokens,
                    "timestamp": datetime.now().isoformat(),
                }

            except Exception as exc:
                logger.error(f"Row {idx} failed: {exc!r}")
                raise

    # -- Orchestrator -------------------------------------------------------

    async def run_pipeline(self, sample_size: Optional[int] = None) -> None:
        """Execute the full extraction pipeline with idempotent checkpointing.

        Parameters
        ----------
        sample_size : int, optional
            If provided, process only the first *sample_size* rows (useful for
            debugging and cost estimation).
        """
        # --- 1. Data ingestion ---------------------------------------------
        reader = pd.read_excel if self.input_file.endswith(".xlsx") else pd.read_csv
        df = reader(self.input_file)
        logger.info(f"Loaded {len(df):,} records from '{self.input_file}'.")

        if sample_size is not None:
            df = df.head(sample_size)
            logger.info(f"Subsetting to first {sample_size:,} records.")

        # --- 2. Idempotency: identify already-processed rows ---------------
        output_file = os.path.join(self.output_dir, "analysis_results_master.csv")
        processed_indices: set = set()

        if os.path.exists(output_file):
            try:
                existing = pd.read_csv(output_file)
                if "original_index" in existing.columns:
                    processed_indices = set(existing["original_index"].unique())
                    logger.info(
                        f"Checkpoint detected: {len(processed_indices):,} rows "
                        f"already processed. Resuming."
                    )
            except Exception:
                logger.warning("Existing output file unreadable. Starting fresh.")

        # --- 3. Build task queue -------------------------------------------
        tasks = []
        for idx, row in df.iterrows():
            if idx not in processed_indices:
                tasks.append(self._analyze_single_row(idx, row))

        if not tasks:
            logger.info("All records already processed. Nothing to do.")
            return

        n_tasks = len(tasks)
        logger.info(
            f"Queuing {n_tasks:,} tasks "
            f"(concurrency={PipelineConfig.MAX_CONCURRENCY}, "
            f"batch_size={PipelineConfig.BATCH_SIZE})."
        )

        # --- 4. Batched execution with incremental persistence -------------
        batch_size = PipelineConfig.BATCH_SIZE
        file_exists = os.path.exists(output_file)

        for batch_idx in range(0, n_tasks, batch_size):
            batch = tasks[batch_idx : batch_idx + batch_size]
            batch_num = batch_idx // batch_size + 1

            results = await tqdm.gather(
                *batch, desc=f"Batch {batch_num}"
            )
            valid = [r for r in results if r is not None]

            if valid:
                pd.DataFrame(valid).to_csv(
                    output_file,
                    mode="a",
                    header=not file_exists,
                    index=False,
                )
                file_exists = True

            logger.info(
                f"Batch {batch_num} complete "
                f"({len(valid)}/{len(batch)} succeeded). "
                f"Cumulative cost: ${self.total_cost:.4f}"
            )

        # --- 5. Summary ----------------------------------------------------
        logger.info(
            f"Pipeline finished. "
            f"Total records processed: {n_tasks:,}. "
            f"Estimated total cost: ${self.total_cost:.4f}."
        )


logger.info("Pipeline class defined.")

---

## 6. Execution & Demonstration

The cell below generates a small synthetic dataset and runs the pipeline end-to-end. This allows reviewers and collaborators to verify the pipeline's functionality without access to the full proprietary dataset or incurring significant API costs.

> **Note:** Set `OPENAI_API_KEY` in your environment or `.env` file before execution. Without a valid key, the demo will exit gracefully.

In [ ]:
# =============================================================================
# Section 6: Execution & Demonstration
# =============================================================================


async def run_demo() -> None:
    """End-to-end demonstration using synthetic data."""

    # --- Synthetic dataset -------------------------------------------------
    mock_records = {
        "Title": ["Inception", "The Room", "The Godfather"],
        "Director": ["Christopher Nolan", "Tommy Wiseau", "Francis Ford Coppola"],
        "Writers": [
            "Christopher Nolan",
            "Tommy Wiseau",
            "Mario Puzo / Francis Ford Coppola",
        ],
        "Release_Year": [2010, 2003, 1972],
        "Budget": [160_000_000, 6_000_000, 6_000_000],
        "Opening_Weekend": [62_785_337, 1_800, 0],
        "Gross_Worldwide": [836_800_000, 4_993_000, 246_100_000],
        "IMDb_Rating": [8.8, 3.6, 9.2],
        "Language": ["English", "English", "English"],
        "Country": ["United States", "United States", "United States"],
        "Production_Companies": [
            "Warner Bros. / Legendary Entertainment",
            "Wiseau-Films",
            "Paramount Pictures / Alfran Productions",
        ],
        "Comments": [
            (
                "A masterpiece of mind-bending storytelling. The layered dream "
                "sequences are visually stunning, and Nolan's direction is nothing "
                "short of genius. Hans Zimmer's score elevates every scene."
            ),
            (
                "This is unironically the worst movie I have ever seen. The acting "
                "is wooden, the dialogue is incomprehensible, and the plot makes "
                "absolutely no sense. A complete disaster from start to finish."
            ),
            (
                "An offer you can't refuse. Brando's performance defines an entire "
                "genre. The pacing, the cinematography, and Coppola's direction are "
                "all flawless. A towering achievement in American cinema."
            ),
        ],
    }

    df_demo = pd.DataFrame(mock_records)
    input_path = "demo_dataset.xlsx"
    df_demo.to_excel(input_path, index=False)
    logger.info(f"Synthetic dataset created: {len(df_demo)} records -> '{input_path}'")

    # --- Pre-flight check --------------------------------------------------
    if not os.getenv("OPENAI_API_KEY"):
        logger.warning(
            "OPENAI_API_KEY not found in environment. "
            "Skipping API calls. Set the key and re-run."
        )
        return

    # --- Run pipeline ------------------------------------------------------
    researcher = MovieReviewResearcher(
        input_file=input_path,
        output_dir="./demo_results",
    )
    await researcher.run_pipeline()

    # --- Display results ---------------------------------------------------
    result_path = "./demo_results/analysis_results_master.csv"
    if os.path.exists(result_path):
        df_result = pd.read_csv(result_path)
        display_cols = [
            "Title",
            "sentiment_score",
            "primary_emotion",
            "emotion_keywords",
            "review_focus",
            "request_cost_usd",
        ]
        existing_cols = [c for c in display_cols if c in df_result.columns]
        print("\n" + "=" * 70)
        print("  RESULTS PREVIEW")
        print("=" * 70)
        display(df_result[existing_cols])
    else:
        logger.warning("No output file generated.")


# Execute in Jupyter
await run_demo()

---

## 7. Notes on Reproducibility

**Determinism.** Language model outputs are inherently stochastic. We mitigate this by setting the temperature to 0.2 and pinning the model version. Nonetheless, exact numerical reproducibility across runs is not guaranteed by the OpenAI API. For the analyses reported in the paper, all sentiment scores were extracted in a single batch run and the resulting CSV was frozen as the canonical dataset.

**Validation.** To assess the reliability of GPT-4o scores, the paper validates them against (i) human-annotated sentiment from trained evaluators, (ii) TextBlob-based lexical sentiment, and (iii) five alternative deep learning architectures (BERT, LSTM, CNN, Bi-LSTM, TIIF). See Tian, Xie & Zhang (2026), Sections 4.3 and 4.4.3 for details.

**Cost management.** At current pricing (\$0.005/1K input, \$0.015/1K output tokens), processing the full corpus of ~247,850 reviews costs approximately \$300-400 USD. The incremental checkpointing mechanism ensures that interrupted runs can be resumed without re-processing completed records.

**Ethical considerations.** All review data were obtained from publicly accessible platforms (IMDb, TMDB) in compliance with their terms of service. No personally identifiable information is retained beyond publicly displayed usernames.

---

*End of notebook.*